# Free-Droid (Szabi) — v5 fine-tune (pozicionális tool-nyelvtan)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

**Mi új a v4-hez képest — a lever a NYELVTAN, nem a hiperparaméter:**
- A v4 benchmark (2026-07-03) igazolta: a fine-tune azt hozza, amiért van (Yotengrit-mélység,
  magyar árnyalat), egy kis koherencia-áron — a `--preset gentle` már v4-ben is futott, a
  hiperparaméter tehát **nem** a lever, marad gentle.
- **Új tool-nyelvtan:** `fn(key="value")` → **pozicionális** (`<tool>move forward 2</tool>`,
  `<tool>turn left 90</tool>`, `<tool>stop</tool>`). Ugyanaz a **681** példa / 613-68 split — csak a
  tool-hívások szintaxisa migrált. A kis modell ezt megbízhatóbban emittálja, és a Python parser
  (`freedroid.tools.parser.parse_tools`) triviálisan, egyértelműen parse-olja.
- **system_prompt.txt:** már a pozicionális formátum-sort tartalmazza — a `finetune.py` ebből tanít,
  így a v5 a parser-barát nyelvtant tanulja.
- **Benchmark oldal:** a judge scorer (`TOOL_RE`) szigorítva — a v5 `tool_calling` oszlopa lesz az
  **első mérvadó** tool-mérés (a régi scorer a `fn()`-t is 5-özte).
- **Recept változatlan:** `--preset gentle` (lr 5e-5, 1 epoch, r/alpha 8, dropout 0.05).

**Először: Runtime → Change runtime type → T4 GPU.**

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## Repo klónozása

A pozicionális-nyelvtanú dataset + az új system_prompt.txt a **`main`-en** van (a
`feature/simplify-tool-grammar` PR mergelve). A notebook a main-t klónozza.
(Ha még merge ELŐTT futtatnád, írd át a lenti `-b main`-t a feature-branch nevére:
`-b feature/simplify-tool-grammar`.)

In [ ]:
# 2. Repo a main-ről (pozicionális dataset + új system_prompt + finetune.py), majd be a training/-be.
!git clone --depth 1 -b main https://github.com/pits2022/free-droid.git
%cd free-droid/training
# Guard: 681 példa ÉS pozicionális nyelvtan (nincs régi fn() forma) ÉS a system_prompt frissült.
!python -c "import json, re; d=json.load(open('dataset/freedroid_full.json')); t=[x for x in d if '<tool>' in x['output']]; c=[x for x in t if x['output'].count('<tool>')>=2]; fn=[x for x in t if re.search(r'<tool>\s*[a-z_]+\(', x['output'])]; sp=open('system_prompt.txt', encoding='utf-8').read(); assert len(d)==681, f'VART 681 pelda, de {len(d)} — rossz branch?'; assert not fn, f'REGI fn() nyelvtan {len(fn)} peldaban — a main meg nem a v5 (pozicionalis) datasetet klonozta (merge kell)?'; assert 'move forward 2' in sp, 'system_prompt.txt nem a pozicionalis formatum-sort tartalmazza — merge kell'; print(f'OK pozicionalis | peldak: {len(d)} | tool: {len(t)} ({100*len(t)/len(d):.1f}%) | osszetett: {len(c)}')"
!wc -l dataset/train.jsonl dataset/val.jsonl

## Edge modell — Llama 3.2 3B (offline fallback)

A Pi 5-ön, teljesen offline futó modell. Q4_K_M (edge) + Q8_0 (cloud) GGUF export.
Kimenet: `outputs/llama3.2-3b-v5/`.

In [ ]:
!python finetune.py --variant llama --preset gentle --tag v5

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A CAX31-en CPU-n futó nagyobb modell — a magyar/persona minőséget a méret hozza. A `llama8b` variáns
T4-biztos (batch=1, grad_accum=8) és csak Q4_K_M-et exportál. Kimenet: `outputs/llama3.1-8b-v5/`.

In [ ]:
!python finetune.py --variant llama8b --preset gentle --tag v5

## Next

- Kimenetek: `training/outputs/<variant>-v5/` — `gguf-q4_k_m` (edge), `gguf-q8_0` (cloud), `lora-adapter`.
  Töltsd le (vagy push HF Hubra) — git-ignorálva.
- **Ollama Modelfile (v5):** a `SYSTEM` blokk = az **új** `training/system_prompt.txt` (a pozicionális
  formátum-sorral!), `PARAMETER temperature 0.7`, `num_ctx 2048`, Llama-3 `TEMPLATE` + stop tokenek —
  az Unsloth GGUF NEM ágyazza be a sablont. (Minta: `tests/v4/**/Modelfile_*`, csak a SYSTEM-et
  cseréld a friss system_prompt.txt-re.) `ollama create szabi-3b-v5 -f Modelfile` / `szabi-8b-v5`.
- **Benchmark:** `run_benchmark.py --models szabi-8b-v5 llama3.1:8b szabi-3b-v5 llama3.2:3b --rag`,
  majd `judge_benchmark.py`. A fő kérdés: **a pozicionális nyelvtan behozta-e a tool_calling-ot** — és
  most a judge scorer is mérvadó (a `fn()`-t már nem fogadja el). Persona/koherencia: ne essen a v4 alá.
- **Kötelező red-team pass** (provokatív / offtopic) a demó előtt.
- A tényeket futásidőben a `freedroid.rag` (offline BM25) adja, nem a fine-tune.